In [1]:
import sys
from pathlib import Path

project_root = Path(
    r"C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation"
)
module_dir = str(project_root / "tools" / "scientific_code" / "PPREstimation")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import pandas as pd
from PPRCalculator import PPRCalculator
from ModelData import ModelData

# Loading

In [2]:
model_path = (
    "../model.json"
)

model = ModelData(
    model_path,
    model_name="Humboldt Current",
    model_year="1995",
)

# Inspect or edit these before constructing the calculator:
# model.groups_data
# model.DC
# model.det_fate

In [3]:
calculator = PPRCalculator.from_modeldata(
    modeldata=model,
    underdetermined=True,     # Estimate missing flows using LIM if True
    zero_catch=True,           # Replace missing catch with zero
    zero_biomass_accum=False,   # Replace missing biomass accumulation with zero
    default_gs=True,           # Supply GS=0.2 for regular groups missing GS
    weight_flow=1.0,           # LIM minimum-flow penalty weight
    weight_guess=1.0,          # LIM guess-deviation penalty weight
    normalize_DC=True,         # Normalize consumer diet sums if True
    DC_tol=0.001,             # Allowed deviation of diet sums from 1
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\ModelData.py:663: RuntimeWarning: 3 consumer group(s) have a diet composition (including diet_import) that does not sum to 1 (tol=0.001): 34 (Green sea turtle)=1.0020, 20 (Large hake )=1.0090, 16 (Chub mackerel)=0.9952.
  warnings.warn(f"{len(bad)} consumer group(s) have a diet composition (including diet_import) "
C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:328: RuntimeWarning: Diet normalization compensated through runtime biomass accumulation for 9 Regular/PP group(s); computational closure, not observed biomass change. EE, M0 and detritus flows retained; see diet_normalization_balance_ledger.
  warnings.warn(


In [4]:
balanced, p, q = calculator.is_model_balanced()
balanced

True

In [5]:
100 * ((p - calculator.p) / calculator.p).rename(index=calculator.seq2name)

group_seq
diet_import                 0.000000e+00
Benthic detritus            0.000000e+00
Pelagic detritus            0.000000e+00
Fishery offal                        NaN
Anchovy eggs                0.000000e+00
Leatherback turtle          0.000000e+00
Green sea turtle            0.000000e+00
Cetaceans                   0.000000e+00
Pinnipeds                   0.000000e+00
Seabirds                    0.000000e+00
Chondrichthyans             3.280006e-06
Catfish                     2.289544e-07
Sea robin                  -7.431666e-07
Medium sciaenids            1.366557e-06
Medium demersal fish        1.519081e-06
Conger                     -8.180733e-05
Butter fishes               7.432482e-07
Benthic elasmobranchs      -1.632514e-06
Small demersal fish        -1.835168e-06
Flatfish                    4.279212e-06
Large hake                 -2.613639e-05
Medium hake                -4.120708e-04
Small hake                  9.101013e-07
Other large pelagic fish   -3.578358e-04
Chub m

# DC

In [8]:
# model.DC.sum(axis=1).rename(index=model.seq2name).round(decimals=7)
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).columns
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).loc['Pelagic sharks and rays']

In [9]:
# calculator.get_DC().sum(axis=1).round(decimals=3)

# Groups Data

In [10]:
# model.groups_data.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

In [12]:
# calculator.get_groups_df().rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

# Diagnostics

In [28]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='GE',
    return_sppr=True,
    det_collapse_mode='auto'
)

In [31]:
sppr = PPRCalculator.rename_results(sppr, calculator.seq2name)
sppr.sum(axis=1).sort_values(ascending=False)

Seabirds                    203171.806955
Cetaceans                    89646.629856
Pinnipeds                    42301.391245
Chondrichthyans               3854.971920
Large hake                    3745.912176
Conger                        2740.205620
Flatfish                      1576.278141
Leatherback turtle            1405.727775
Other large pelagic fish      1309.011574
Green sea turtle              1193.636088
Chub mackerel                  900.318696
Medium hake                    878.495948
Benthic elasmobranchs          873.473966
Small hake                     823.985992
Jumbo squid                    660.655578
Medium sciaenids               654.261493
Horse mackerel                 581.753508
Catfish                        544.146639
Medium demersal fish           457.651705
Mesopelagic fish               233.459173
Sea robin                      199.133978
Sardine                        163.037398
Butter fishes                  150.245209
Other small pelagic fish       131

In [22]:
result['model_input']

{'status': 'WARN',
 'is_model_balanced': True,
 'p_max_rel_residual': 9.330351026379287e-06,
 'q_max_rel_residual': 9.330351026379287e-06,
 'dc_rows_sum_to_1': True,
 'dc_max_deviation': 2.220446049250313e-16,
 'n_negative_catch': 0,
 'n_zero_catch': 22,
 'total_catch': 31.2882064180725,
 'has_catch': True,
 'has_ee_issues': True,
 'n_ee0': 5,
 'n_ee_marginal': 1,
 'n_ee_gt_1': 0,
 'ee0_groups': [35, 34, 33, 32, 31],
 'ee_marginal_groups': [7]}

In [23]:
result['divergence']

{'status': 'WARN',
 'solve_error': None,
 'b': 0.0,
 'b_converges': True,
 'rho_living': 0.7316477059387119,
 'living_converges': True,
 'sppr_det': {36: 0.0, 37: 0.0, 38: 0.30428520886027294, 39: 0.0},
 'max_sppr_det': 0.30428520886027294,
 'max_sppr_group': {'seq': 25,
  'tl': 4.1173056366305385,
  'sppr': 447435.87436565995,
  'inv_te': 590.4578556431102},
 'max_tl_group': {'seq': 30,
  'tl': 4.44723808533033,
  'sppr': 7573.269897306671,
  'inv_te': 11.815484985703286},
 'n_negative_sources': 0,
 'expect_negatives': False,
 'near_singular_te': [7]}

In [24]:
result['balance']

{'status': 'FAIL',
 'is_balanced': False,
 'inflow': 11310.477371346196,
 'outflow': 9740.765427533966,
 'rel_gap': 0.13878388084564106}

In [25]:
result['footprint']

{'ppr_all': 8191.161662001676,
 'ppr_inner': 8156.883789816555,
 'ppr_pp_only': 8156.883789816555,
 'npp': 11273.056447505951,
 'ppr2npp': 0.7235734006832887,
 'ppr2npp_pp_only': 0.7235734006832887}

In [26]:
result['config']

{'TE_option': 'TE',
 'det_open_mode': 'none',
 'det_theta': 1.0,
 'det_external_sppr': 0.0,
 'det_collapse_mode': 'auto',
 'explicit_TE': False,
 'method': None,
 'would_pool': False,
 'model': 'Humboldt Current (1995)'}

In [27]:
result['warnings']

["5 group(s) with EE=0 (all production is non-predatory death): 35 (Leatherback turtle), 34 (Green sea turtle), 33 (Cetaceans), 32 (Pinnipeds), 31 (Seabirds). Under TE_option='TE' their TE row is 0, which severs them from the nullspace and leaks the PP they consumed",
 '1 group(s) with 0 < EE < 0.001: 7 (Chrysaora plocamia). Their SPPR ~ 1/te is near-singular',
 "TE_option='TE' has no detritus recycling matrix: b reported as 0.0 (mortality-derived SPPR is written off as lost)",
 'rho(A_LL)=0.7316 is within 0.3 of divergence (max SPPR 4.47e+05 at group 25 (Conger), TL 4.12)',
 '1 group(s) have near-zero TE (SPPR ~ 1/te is near-singular): [7]',
 'PP balance gap 13.878% exceeds fail threshold 5.0%']